# Lesson 6 — Sanity-check the three VAD models

Before training, we verify three things about each model:

1. **Shapes are correct** — given an `(B, 80, T)` input, the model returns `(B, T)` logits.
2. **Parameter counts match expectations** — if a model has way more or fewer parameters than expected, something is misconfigured.
3. **One forward pass on the GPU works** and produces non-degenerate outputs (not all zeros, not NaN).

We're not training yet — that's lesson 7. This is the pre-flight check.


In [ ]:
import sys
sys.path.insert(0, '/workspace')

import torch
import torch.nn as nn

from vad_models import build_model, count_parameters

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"  GPU: {torch.cuda.get_device_name(0)}")


## 1. Instantiate all three models and check parameter counts

In [ ]:
for name in ['mlp', 'cnn', 'crnn']:
    model = build_model(name)
    n = count_parameters(model)
    print(f"  {name.upper():6}: {n:>8,} parameters")


**What to expect:**

```
  MLP   :   ~18,700 parameters
  CNN   :   ~67,200 parameters
  CRNN  :   ~96,500 parameters
```

If your numbers are wildly off (10x bigger or smaller), check whether you accidentally changed defaults in `vad_models.py`.

For reference: Silero VAD is ~1.8M parameters, but heavily optimized. Our CRNN at ~96K is intentionally tiny to show how much you can do with very little. We'll scale up later if needed.

---

## 2. Forward-pass sanity check

Generate a fake input of the right shape, push it through each model, verify the output shape and that nothing is NaN.


In [ ]:
# Simulated batch: 8 clips, 80 mel bins, 400 frames (= 4 seconds at 10ms hop)
B, M, T = 8, 80, 400
fake_input = torch.randn(B, M, T, device=device)

for name in ['mlp', 'cnn', 'crnn']:
    model = build_model(name).to(device)
    model.eval()
    with torch.no_grad():
        out = model(fake_input)
    print(f"{name.upper():6}: in {tuple(fake_input.shape)} → out {tuple(out.shape)}, "
          f"range [{out.min().item():.3f}, {out.max().item():.3f}], "
          f"any NaN: {torch.isnan(out).any().item()}")


**What to expect:**

```
MLP   : in (8, 80, 400) → out (8, 400), range [...], any NaN: False
CNN   : in (8, 80, 400) → out (8, 400), range [...], any NaN: False
CRNN  : in (8, 80, 400) → out (8, 400), range [...], any NaN: False
```

All three should produce `(8, 400)` — same batch size, same time length as input. Logit ranges with random input typically land in `[-3, 3]`. If the range is huge (`[-1000, 1000]`) something is unstable; if it's NaN, we have a bug.

---

## 3. End-to-end: run a real sample from the dataset through each model

This connects lesson 5 and lesson 6. Pull one sample from the dataset, push it through each (untrained) model, and just verify the plumbing works.


In [ ]:
from vad_dataset import VADDataset

ds = VADDataset(
    speech_dir="/workspace/data/LibriSpeech/train-clean-100",
    noise_dir="/workspace/data/musan/noise",
    music_dir="/workspace/data/musan/music",
    clip_seconds=4.0,
    seed=0,
)

feat, lab = ds[0]
print(f"Feature shape from dataset: {feat.shape}")
print(f"Label shape from dataset:   {lab.shape}")

# Move to GPU and add batch dim
feat = feat.unsqueeze(0).to(device)
lab  = lab.unsqueeze(0).to(device)

for name in ['mlp', 'cnn', 'crnn']:
    model = build_model(name).to(device)
    model.eval()
    with torch.no_grad():
        logits = model(feat)
    probs = torch.sigmoid(logits)
    print(f"{name.upper():6}: logits {tuple(logits.shape)}, "
          f"mean prob (untrained): {probs.mean().item():.3f}")


**What to expect:**

All three models should output `(1, T)` shape, with mean probability around 0.5 (untrained → roughly random predictions). After training, the mean probability for clips with mostly speech should rise to ~0.7+.

---

## 4. Visualize untrained predictions (mostly for fun)

Plot the dataset spectrogram, the ground-truth labels, and each model's untrained predictions. Untrained predictions should look like noise. After training (next lesson), they'll start tracking the labels.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import librosa.display

feat_np = feat.squeeze().cpu().numpy()
lab_np  = lab.squeeze().cpu().numpy()
times = np.arange(lab_np.shape[0]) * 160 / 16000

fig, axes = plt.subplots(5, 1, figsize=(14, 11), sharex=True)

# Top: log-mel spectrogram
librosa.display.specshow(feat_np, sr=16000, hop_length=160,
                         x_axis='time', y_axis='mel', cmap='magma', ax=axes[0])
axes[0].set_title("Input log-mel spectrogram")

# Ground-truth labels
axes[1].fill_between(times, 0, lab_np, step='mid', alpha=0.5, color='C2')
axes[1].set_title("Ground-truth speech labels")
axes[1].set_ylim(-0.1, 1.1)

# Each model's untrained output
for ax, name in zip(axes[2:], ['mlp', 'cnn', 'crnn']):
    model = build_model(name).to(device)
    model.eval()
    with torch.no_grad():
        probs = torch.sigmoid(model(feat)).squeeze().cpu().numpy()
    ax.plot(times, probs, linewidth=1, color='C1')
    ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.5)
    ax.set_title(f"{name.upper()} predictions (untrained — should look like noise)")
    ax.set_ylim(-0.1, 1.1)

axes[-1].set_xlabel("Time (s)")
plt.tight_layout()
plt.show()


**What to expect:**

- Spectrogram and ground-truth labels look like a normal training sample.
- All three model predictions hover around 0.5 with random wiggles. **They don't track the labels yet — they haven't been trained.**

This baseline is important to remember. When you re-run this notebook after lesson 7's training, the predictions should snap onto the labels. The difference between "random noise" here and "tracks the labels" then is what training accomplishes.

---

## 5. (Optional) Speed test — frames per second on the GPU

Just so you know what to expect during training.


In [ ]:
import time

B, M, T = 64, 80, 400  # batch 64, 4-second clips
fake = torch.randn(B, M, T, device=device)

for name in ['mlp', 'cnn', 'crnn']:
    model = build_model(name).to(device)
    model.train()  # closer to training-time speed
    optimizer = torch.optim.Adam(model.parameters())

    # warmup
    for _ in range(3):
        optimizer.zero_grad()
        out = model(fake)
        loss = out.mean()
        loss.backward()
        optimizer.step()

    # measure
    if device.type == 'cuda':
        torch.cuda.synchronize()
    t0 = time.time()
    n_iter = 50
    for _ in range(n_iter):
        optimizer.zero_grad()
        out = model(fake)
        loss = out.mean()
        loss.backward()
        optimizer.step()
    if device.type == 'cuda':
        torch.cuda.synchronize()
    dt = time.time() - t0

    samples_per_sec = n_iter * B / dt
    frames_per_sec  = samples_per_sec * T
    print(f"{name.upper():6}: {samples_per_sec:>8.0f} samples/sec, "
          f"{frames_per_sec/1e6:>5.2f} M frames/sec")


**On an L40S you should see something like:**

```
MLP   :   ~30000 samples/sec   12.00 M frames/sec
CNN   :   ~15000 samples/sec    6.00 M frames/sec
CRNN  :    ~8000 samples/sec    3.20 M frames/sec
```

These are *training* throughput numbers (forward + backward + step). The CRNN is slowest because GRUs don't parallelize over time the way CNNs do — each step depends on the previous. But 8K samples/sec is plenty fast.

The data loader from lesson 5 was around 1-4K samples/sec, so the GPU will be **data-bound** for the CNN and CRNN. We'll address this when training takes shape — easiest fix is caching mel features to disk so the CPU doesn't redo the same work every epoch.

---

## You're done with lesson 6 when…

- All three models instantiate without errors
- Parameter counts match expectations
- Forward passes produce the right output shape
- Untrained predictions look like random noise around 0.5

Ping me with one of:
- **"All three models work, take me to lesson 7 (training loop)"**
- **"Shape mismatch in model X"** — paste the error
- **"My CRNN has 700K parameters not 75K"** — easy fix, somebody scaled up the defaults

Lesson 7 is the training loop. After that, you train these three models, compare them, and have a working VAD by the end of the next session.
